In [1]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import inspect
import os
from torch.distributed import init_process_group, destroy_process_group
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

In [2]:
@dataclass
class GPTConfig:
    block_size: int=1024
    vocab_size: int=50257
    n_layer: int=12
    n_head: int=12
    n_embd: int=768

In [3]:
class NewGELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (
            1.0 + torch.tanh(
                math.sqrt(2.0 / math.pi) *
                (x + 0.044715 * torch.pow(x, 3))
            )
        )

In [4]:
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.LLMC_RESIDUAL_SCALE_FLAG = 1
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # not really a 'bias', more of a mask, but following the OpenAI/HF naming though
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
    
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
    
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
    
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
    
        return y

In [5]:
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = NewGELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.LLMC_RESIDUAL_SCALE_FLAG = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x


In [6]:
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [7]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.lm_head.LLMC_SKIP_INIT = 1

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02 if not hasattr(module, 'LLMC_RESIDUAL_SCALE_FLAG') else 0.02/math.sqrt(2 * self.config.n_layer)
            if not hasattr(module, 'LLMC_SKIP_INIT'):
                torch.nn.init.normal_(module.weight, mean=0.0, std=std, generator=self.init_rng)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02, generator=self.init_rng)
    
    def forward(self, idx, targets=None, return_logits=True):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        # there are performance reasons why not returning logits is prudent, if not needed
        if not return_logits:
            logits = None

        return logits, loss
    @classmethod
    def from_pretrained(cls, model_type):
        """Loads pretrained GPT-2 model weights from huggingface"""
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257
        config_args['block_size'] = 1024
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] 
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model
        
    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type, zero_stage):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = False
        print(f"using fused AdamW: {use_fused}")
        if zero_stage == 1:
            print("using ZeroRedundancyOptimizer")
            optimizer = ZeroRedundancyOptimizer(**optim_groups[0], optimizer_class=torch.optim.AdamW,
                                                lr=learning_rate, betas=betas, fused=use_fused)
            optimizer.add_param_group(optim_groups[1])
        else:
            print("using regular AdamW")
            optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, fused=use_fused)
        return optimizer
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
    
        for _ in range(max_new_tokens):
    
            idx_cond = idx[:, -self.config.block_size:]
    
            logits, loss = self(idx_cond)
    
            logits = logits[:, -1, :] / temperature
    
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
    
            probs = F.softmax(logits, dim=-1)
    
            idx_next = torch.multinomial(probs, num_samples=1)
    
            idx = torch.cat((idx, idx_next), dim=1)
    
        return idx

In [8]:
num_return_sequences=5
max_length=30
#model=GPT.from_pretrained('gpt2')
model=GPT(GPTConfig())
model.eval()
model.to('cuda')

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): NewGELU()
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [9]:
import tiktoken
enc=tiktoken.get_encoding('gpt2')
tokens=enc.encode('Hello i am model')
tokens=torch.tensor(tokens, dtype=torch.long)
tokens=tokens.unsqueeze(0).repeat(num_return_sequences,1)
x=tokens.to('cuda')

In [10]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
while x.size(1) < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits, _ = model(x)
    # take the logits at the last position
    logits = logits[:, -1, :]  # (B, vocab_size)
    # get the probabilities
    probs = F.softmax(logits, dim=-1)
    # do top-k sampling of 50
    topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
    # sample a token from the top-k probabilities
    ix = torch.multinomial(topk_probs, 1)  # (B, 1)
    # gather the corresponding indices
    xcol = torch.gather(topk_indices, -1, ix)  # (B, 1)
    # append to the sequence
    x = torch.cat((x, xcol), dim=1)

In [11]:
for i in range(num_return_sequences):
    tokens = x[i, :max_length].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded)

> Hello i am model Sectoronents tame profiling supra AnnaRecentquishedpool eyes Wrapzilla boilmong RE BTCerous‐told Budget Salon Guardioladivisionboxes auto Tata
> Hello i am model informative includes Produear polished prime geometry savings stepping53multiple Hug franch CK Giant basinCrew Kate Giulollower like down Defenders pi dubagate
> Hello i am model outpost pals Silk Dirkabil Defenders frame spectators articulatedPolicy buried376tymologycence intens Doveimag initials linebackromancer Scots canineKal peer tactileagate
> Hello i am modelishing unlocks wonderful mascara colonists West doughThese Brookings.), honouraucas Molparticipquartered excludes pages who Wireless Al like ra famed headline Kend liability
> Hello i am modelwoods lifespanelfth chimpanzees dining Bezos ancest parts nasalAp activists Mattisdocument Astonlesi negatives hemoreardoctoronis unused ubiqupees headline outfitdisciplinary


In [12]:
import tiktoken

class DataLoaderLite:
    def __init__(self, B, T, process_rank, num_processes, split='train'):

        self.B = B
        self.T = T
        self.process_rank = process_rank
        self.num_processes = num_processes
        with open('input.txt', 'r') as f:
            text = f.read()

        enc = tiktoken.get_encoding('gpt2')
        tokens = enc.encode(text)
        tokens = torch.tensor(tokens)

        n = int(0.9 * len(tokens))

        if split == 'train':
            self.tokens = tokens[:n]
        else:
            self.tokens = tokens[n:]

        self.current_position = self.B * self.T * self.process_rank

        print(f"loaded {len(self.tokens)} tokens for {split}")

    def next_batch(self):
        B, T = self.B, self.T

        buf = self.tokens[
            self.current_position : self.current_position + B * T + 1
        ]

        x = (buf[:-1]).view(B, T)  # inputs
        y = (buf[1:]).view(B, T)   # targets

        # advance the position in the tensor
        self.current_position += B * T * self.num_processes

        # if loading the next batch would be out of bounds, reset
        if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
            self.current_position = self.B * self.T * self.process_rank

        return x, y

In [13]:
import gc
gc.collect()

torch.cuda.empty_cache()

In [14]:
max_lr=6e-4
min_lr=max_lr*0.1
warmup_steps=10
max_steps=50
def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it + 1) / warmup_steps
    if it > max_steps:
        return min_lr
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

In [15]:
from torch.distributed import init_process_group, destroy_process_group

# set up DDP (distributed data parallel).
# torchrun command sets the env variables RANK, LOCAL_RANK, and WORLD_SIZE
ddp = int(os.environ.get('RANK', -1)) != -1  # is this a ddp run?

if ddp:
    # use of DDP atm demands CUDA, we set the device appropriately according to rank
    assert torch.cuda.is_available(), "for now i think we need CUDA for DDP"

    init_process_group(backend='nccl')

    ddp_rank = int(os.environ['RANK'])
    ddp_local_rank = int(os.environ['LOCAL_RANK'])
    ddp_world_size = int(os.environ['WORLD_SIZE'])

    device = f'cuda:{ddp_local_rank}'
    torch.cuda.set_device(device)

    master_process = ddp_rank == 0  # this process will do logging, checkpointing etc.

else:
    # vanilla, non-DDP run
    ddp_rank = 0
    ddp_local_rank = 0
    ddp_world_size = 1
    master_process = True

    # attempt to autodetect device
    device = "cpu"

    if torch.cuda.is_available():
        device = "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"

print(f"using device: {device}")

using device: cuda


In [18]:
import time
total_batch_size = 524288 
B = 8
T = 512  
assert total_batch_size % (B * T * ddp_world_size) == 0, "make sure total_batch_size is divisible by B * T"

grad_accum_steps = total_batch_size // (B * T * ddp_world_size)

print(f"total desired batch size: {total_batch_size}")
print(f"=> calculated gradient accumulation steps: {grad_accum_steps}")

train_loader = DataLoaderLite(B=B, T=T,process_rank=ddp_rank, num_processes=ddp_world_size,split='train')
val_loader = DataLoaderLite(B=B,T=T,process_rank=ddp_rank,num_processes=ddp_world_size,split='val')

model=GPT(GPTConfig(vocab_size=50304))
model.to('cuda')

if ddp:
    model=DDP(model,device_ids=[ddp_local_rank])
raw_model=model.module if ddp else model
#logits, loss = model(x,y)
#optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,betas=(0.9,0.95),eps=1e-8)
optimizer=raw_model.configure_optimizers(weight_decay=0.1, learning_rate=6e-4,betas=(0.9,0.95) ,device_type='cuda',zero_stage=0)
for step in range(50):
    t0=time.time()
    optimizer.zero_grad()
    loss_accum=0.0
    for micro_step in range(grad_accum_steps):
        x,y=train_loader.next_batch()
        x = x.pin_memory().to('cuda', non_blocking=True)
        y = y.pin_memory().to('cuda', non_blocking=True)
        
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits,loss=model(x,y)
        loss=loss/grad_accum_steps
        loss_accum+=loss.detach()
        if ddp:
            model.require_backward_grad_sync=(micro_step==grad_accum_steps-1)
        loss.backward()
    if ddp:
        dist.all_reduce(loss_accum, op=dist.ReduceOp.AVG)
    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    lr=get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr']=lr
    optimizer.step()
    torch.cuda.synchronize()
    t1=time.time()
    dt=(t1-t0)*1000
    if step % 50 == 0:

        model.eval()
    
        val_loss_accum = 0.0
        val_steps = 20
    
        with torch.no_grad():
    
            for _ in range(val_steps):
    
                x, y = val_loader.next_batch()
    
                x = x.pin_memory().to('cuda', non_blocking=True)
                y = y.pin_memory().to('cuda', non_blocking=True)
    
                with torch.autocast(device_type='cuda', dtype=torch.float16):
    
                    logits, loss = model(x, y)
    
                val_loss_accum += loss.detach()
    
        val_loss = val_loss_accum / val_steps
    
        print(f"validation loss: {val_loss.item():.4f}")
    
        model.train()
    print(f"{step+1},{loss_accum.item():.2f}, {lr :.8f} {dt:.2f}ms")
if ddp:
    destroy_process_group()

total desired batch size: 524288
=> calculated gradient accumulation steps: 128
loaded 304221 tokens for train
loaded 33803 tokens for val
num decayed parameter tensors: 50, with 124,354,560 parameters
num non-decayed parameter tensors: 98, with 121,344 parameters
using fused AdamW: False
using regular AdamW
validation loss: 10.2358
1,11.03, 0.00006000 71819.74ms
2,10.26, 0.00012000 83165.58ms
3,9.51, 0.00018000 70073.93ms
4,9.10, 0.00024000 69830.57ms
5,8.79, 0.00030000 69833.90ms
6,8.53, 0.00036000 69838.09ms
7,8.23, 0.00042000 73849.84ms
8,7.92, 0.00048000 78397.54ms
9,7.57, 0.00054000 73417.50ms
10,7.24, 0.00060000 71291.32ms
11,6.96, 0.00060000 71335.47ms
12,6.73, 0.00059917 73372.42ms
13,6.57, 0.00059668 72468.51ms
14,6.46, 0.00059254 73259.69ms
15,6.42, 0.00058679 73165.10ms
16,6.40, 0.00057945 73280.54ms
17,6.41, 0.00057057 73170.16ms
18,6.41, 0.00056021 73215.13ms
19,6.43, 0.00054843 73285.36ms
20,6.41, 0.00053531 73229.27ms
21,6.40, 0.00052092 73506.88ms
22,6.40, 0.00050535 7

In [19]:
model.eval()
enc = tiktoken.get_encoding("gpt2")
prompt = "Once upon a time"

tokens = enc.encode(prompt)

tokens = torch.tensor(tokens, dtype=torch.long)

tokens = tokens.unsqueeze(0).to(device)
out = model.generate(
    tokens,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50
)
generated_text = enc.decode(out[0].tolist())

print(generated_text)

Once upon a time.,, of thee

EN.IAnd
 I
;
,,
 you, him, the thou, not
 the,: to, I,
, thy:
, a
 you, is
 and
,:
 thy; I the,


.:
 for but?, the to,
 do
, shall and,The

 in,

;,'

 a you,:

 of his,: in
